In [24]:
import os
import contextlib
import io
import pandas as pd

# from bias_bench.test.automated_test import test_crows

directory = "../data/crows_improved/"

to_test_types = ["gender", "race-color", "religion"]
# to_test_languages = ["de_DE", "mt_MT"]
to_test_languages = ["ar_DZ", "ca_ES", "de_DE", "en_US", "es_AR", "fr_FR", "it_IT", "mt_MT", "zh_CN"]

languages = []

for filename in os.listdir(directory):
    filepath = os.path.join(directory, filename)
    if os.path.isdir(filepath):
        languages.append(filename)

In [25]:
files_to_use = {}
df_english = pd.DataFrame()
fr_frensch = pd.DataFrame()

for lang in languages:
    path = directory + lang + "/"
    files = os.listdir(path)
    if len(files) > 1:
        for file in files:
            if "corrected" in file:
                files_to_use[lang] = (path + file)
    else:
        files_to_use[lang] = (path + files[0])

files_to_use

{'mt_MT': '../data/crows_improved/mt_MT/maltese_malta_crowspairs.csv',
 'ar_DZ': '../data/crows_improved/ar_DZ/ar.csv',
 'en_US': '../data/crows_improved/en_US/english_US_crowspairs_corrected.csv',
 'de_DE': '../data/crows_improved/de_DE/german_germany_crowspairs.csv',
 'es_AR': '../data/crows_improved/es_AR/spanish_argentina_crowspairs.csv',
 'fr_FR': '../data/crows_improved/fr_FR/french_france_crowspairs_corrected.csv',
 'zh_CN': '../data/crows_improved/zh_CN/simplified_chinese_china_crowspairs.csv',
 'it_IT': '../data/crows_improved/it_IT/italian_italy_crowspairs.csv',
 'ca_ES': '../data/crows_improved/ca_ES/catalan_spain_crowspairs.csv'}

In [27]:
df_languages_unedited = {}

for file in files_to_use:
    if file not in to_test_languages:
        continue
    df = pd.read_csv(files_to_use[file], sep=None, engine="python", encoding="utf-8-sig")
    if file == "es_AR":
        df_es_AR = df.drop(columns=["20"])
        df_es_AR["bias_type"] = df_es_AR["bias_type_es"].combine_first(df_es_AR["bias_type"])
        df = df_es_AR.drop(columns=["bias_type_es"])
    
    if file == "it_IT":
        print("it")
        df_it_IT = df.drop(columns=["stereoype_comment", "sentence_comment"])
        df_en_US = pd.read_csv(files_to_use["en_US"], sep=None, engine="python", encoding="utf-8-sig")
        df_it_IT = df_it_IT.merge(
            df_en_US[["id", "stereo_antistereo"]],
            on="id",
            how="left"
        )
        print(df_it_IT.columns)
        df = df_it_IT
        
    df = df.rename(columns={df.columns[0]: "id"})
    df_languages_unedited[file] = df

it
Index(['id', 'more_ita', 'less_ita', 'bias_type', 'stereo_antistereo'], dtype='str')


In [28]:
import difflib

df_languages = {}

to_merge_columns = ["sent_more", "sent_less", "stereo_antistereo", "bias_type"]

path_to_safe = ""

# print(df_languages_unedited)

for lang, content in df_languages_unedited.items():
    column_names = content.columns.tolist()

    most_likely_colum = {}

    for column in to_merge_columns:
        # most_likely_colum[column] = difflib.get_close_matches(column + "_" + lang, column_names, n = 1)[0]
        most_likely_colum[difflib.get_close_matches(column + "_" + lang, column_names, n = 1)[0]] = column
    
    df_languages[lang] = df_languages_unedited[lang].filter(items=list(most_likely_colum.keys()))
    df_languages[lang] = df_languages[lang].rename(columns=most_likely_colum)

    df_languages[lang].index.name = "id"
    df_languages[lang].to_csv(directory + f"crows_{lang}.csv")
    
    print(lang, "----------------------------------------")
    print(df_languages[lang].columns)

mt_MT ----------------------------------------
Index(['sent_more', 'sent_less', 'stereo_antistereo', 'bias_type'], dtype='str')
ar_DZ ----------------------------------------
Index(['sent_more', 'sent_less', 'stereo_antistereo', 'bias_type'], dtype='str')
en_US ----------------------------------------
Index(['sent_more', 'sent_less', 'stereo_antistereo', 'bias_type'], dtype='str')
de_DE ----------------------------------------
Index(['sent_more', 'sent_less', 'stereo_antistereo', 'bias_type'], dtype='str')
es_AR ----------------------------------------
Index(['sent_more', 'sent_less', 'stereo_antistereo', 'bias_type'], dtype='str')
fr_FR ----------------------------------------
Index(['sent_more', 'sent_less', 'stereo_antistereo', 'bias_type'], dtype='str')
zh_CN ----------------------------------------
Index(['sent_more', 'sent_less', 'stereo_antistereo', 'bias_type'], dtype='str')
it_IT ----------------------------------------
Index(['sent_more', 'sent_less', 'stereo_antistereo', 'bi